# 电商用户复购预测项目

## 基于Spark MLlib + 深度学习的电商用户消费行为画像与复购预测

---

In [ ]:
# 设置Spark环境变量
import os
os.environ['PYSPARK_PYTHON'] = 'c:\\Users\\MECHREVO\\Desktop\\深度学习\\.venv\\Scripts\\python.exe'
os.environ['PYSPARK_DRIVER_PYTHON'] = 'c:\\Users\\MECHREVO\\Desktop\\深度学习\\.venv\\Scripts\\python.exe'

print('环境变量设置完成')

In [ ]:
# 导入基础库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

# 导入sklearn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print('基础库导入成功')

In [ ]:
# 导入TensorFlow
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print(f'TensorFlow版本: {tf.__version__}')
print('深度学习库导入成功')

In [ ]:
# ========================================
# 第二部分：模拟数据生成
# ========================================

np.random.seed(42)
n_users = 5000
n_records = 50000

# 生成用户数据
users = pd.DataFrame({
    'user_id': np.arange(1, n_users + 1),
    'register_date': pd.date_range('2023-01-01', periods=n_users, freq='H'),
    'age': np.random.randint(18, 65, n_users),
    'gender': np.random.choice(['男', '女'], n_users)
})

# 生成行为记录
behavior_data = pd.DataFrame({
    'user_id': np.random.randint(1, n_users + 1, n_records),
    'product_id': np.random.randint(1, 1001, n_records),
    'category_id': np.random.randint(1, 21, n_records),
    'action_type': np.random.choice([0, 1, 2], n_records, p=[0.6, 0.25, 0.15]),
    'timestamp': pd.date_range('2023-06-01', periods=n_records, freq='T'),
    'price': np.random.uniform(10, 1000, n_records).round(2),
    'quantity': np.random.randint(1, 5, n_records)
})

print(f'用户数据: {len(users)} 条')
print(f'行为记录: {len(behavior_data)} 条')
print('\n行为类型分布:')
print(behavior_data['action_type'].value_counts())
print('\n数据生成完成!')

In [ ]:
# ========================================
# 第三部分：特征工程
# ========================================

# 计算用户特征
user_features = behavior_data.groupby('user_id').agg(
    total_clicks=('action_type', lambda x: (x == 1).sum()),
    total_browses=('action_type', lambda x: (x == 0).sum()),
    total_purchases=('action_type', lambda x: (x == 2).sum()),
    total_amount=('price', lambda x: (x * behavior_data.loc[x.index, 'quantity']).sum()),
    avg_price=('price', 'mean'),
    purchase_days=('timestamp', lambda x: x.dt.date.nunique()),
    first_action=('timestamp', 'min'),
    last_action=('timestamp', 'max'),
    unique_categories=('category_id', 'nunique'),
    unique_products=('product_id', 'nunique')
).reset_index()

# 计算更多特征
user_features['avg_order_value'] = user_features['total_amount'] / user_features['total_purchases'].replace(0, 1)
user_features['click_to_purchase_ratio'] = user_features['total_clicks'] / user_features['total_purchases'].replace(0, 1)
user_features['browse_to_purchase_ratio'] = user_features['total_browses'] / user_features['total_purchases'].replace(0, 1)
user_features['days_since_last_action'] = (pd.Timestamp.now() - user_features['last_action']).dt.days
user_features['active_days'] = (user_features['last_action'] - user_features['first_action']).dt.days

# 合并用户基础信息
user_features = user_features.merge(users, on='user_id', how='left')

print(f'用户特征维度: {user_features.shape}')
print('\n用户特征预览:')
print(user_features.head())

In [ ]:
# ========================================
# 第四部分：标签构造 - 30天复购标签
# ========================================

# 确定时间分割点
cutoff_date = behavior_data['timestamp'].max() - timedelta(days=30)

# 计算用户在30天内是否有购买
recent_purchases = behavior_data[
    (behavior_data['action_type'] == 2) & 
    (behavior_data['timestamp'] > cutoff_date)
]['user_id'].unique()

# 添加复购标签
user_features['repurchase_label'] = user_features['user_id'].isin(recent_purchases).astype(int)

print(f'复购用户比例: {user_features["repurchase_label"].mean():.2%}')
print(f'总用户数: {len(user_features)}')
print(f'复购用户数: {user_features["repurchase_label"].sum()}')
print(f'非复购用户数: {len(user_features) - user_features["repurchase_label"].sum()}')

In [ ]:
# ========================================
# 第五部分：数据预处理
# ========================================

# 选择特征列
feature_cols = [
    'total_clicks', 'total_browses', 'total_purchases',
    'total_amount', 'avg_price', 'purchase_days',
    'unique_categories', 'unique_products',
    'avg_order_value', 'click_to_purchase_ratio',
    'browse_to_purchase_ratio', 'days_since_last_action',
    'active_days', 'age'
]

# 处理缺失值
X = user_features[feature_cols].fillna(0)
y = user_features['repurchase_label']

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 特征标准化
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'训练集: {X_train.shape}')
print(f'测试集: {X_test.shape}')
print('数据预处理完成!')

In [ ]:
# ========================================
# 第六部分：传统机器学习模型
# ========================================

# 逻辑回归
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
y_pred_lr = lr_model.predict(X_test_scaled)
y_prob_lr = lr_model.predict_proba(X_test_scaled)[:, 1]

# 随机森林
rf_model = RandomForestClassifier(
    n_estimators=100, random_state=42,
    max_depth=10, min_samples_split=10
)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

# 评估结果
def evaluate_model(y_true, y_pred, y_prob, model_name):
    accuracy = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_prob)
    recall = recall_score(y_true, y_pred)
    print(f'{model_name}:')
    print(f'  准确率: {accuracy:.4f}')
    print(f'  AUC: {auc:.4f}')
    print(f'  召回率: {recall:.4f}')
    return accuracy, auc, recall

print('=' * 40)
print('模型评估结果')
print('=' * 40)
lr_acc, lr_auc, lr_recall = evaluate_model(y_test, y_pred_lr, y_prob_lr, '逻辑回归')
print()
rf_acc, rf_auc, rf_recall = evaluate_model(y_test, y_pred_rf, y_prob_rf, '随机森林')

In [ ]:
# ========================================
# 第七部分：深度学习模型
# ========================================

# 构建DNN模型
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC()]
)

early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

# 训练模型
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stopping],
    verbose=1
)

# 预测
y_pred_dnn = (model.predict(X_test_scaled) > 0.5).astype(int).flatten()
y_prob_dnn = model.predict(X_test_scaled).flatten()

# 评估
print('\n' + '=' * 40)
dnn_acc, dnn_auc, dnn_recall = evaluate_model(y_test, y_pred_dnn, y_prob_dnn, 'DNN深度学习')

In [ ]:
# ========================================
# 第八部分：模型结果对比
# ========================================

# 对比表格
results = pd.DataFrame({
    '模型': ['逻辑回归', '随机森林', 'DNN深度学习'],
    '准确率': [lr_acc, rf_acc, dnn_acc],
    'AUC': [lr_auc, rf_auc, dnn_auc],
    '召回率': [lr_recall, rf_recall, dnn_recall]
})

print('模型性能对比:')
print(results.round(4))

# 可视化对比
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].bar(results['模型'], results['准确率'], color=['blue', 'green', 'orange'])
axes[0].set_title('准确率对比')
axes[0].set_ylim(0.6, 0.9)

axes[1].bar(results['模型'], results['AUC'], color=['blue', 'green', 'orange'])
axes[1].set_title('AUC对比')
axes[1].set_ylim(0.6, 0.95)

axes[2].bar(results['模型'], results['召回率'], color=['blue', 'green', 'orange'])
axes[2].set_title('召回率对比')
axes[2].set_ylim(0.3, 0.7)

plt.tight_layout()
plt.show()

In [ ]:
# ========================================
# 第九部分：用户分群 - K-Means聚类
# ========================================

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# 使用K-Means聚类
kmeans = KMeans(n_clusters=4, random_state=42)
user_features['cluster'] = kmeans.fit_predict(X)

# 使用PCA可视化
pca = PCA(n_components=2)
pca_result = pca.fit_transform(X)
user_features['pca1'] = pca_result[:, 0]
user_features['pca2'] = pca_result[:, 1]

# 可视化聚类结果
plt.figure(figsize=(10, 8))
sns.scatterplot(
    x='pca1', y='pca2',
    hue='cluster',
    data=user_features,
    palette='viridis',
    s=50,
    alpha=0.7
)
plt.title('用户聚类结果可视化')
plt.xlabel('PCA维度1')
plt.ylabel('PCA维度2')
plt.legend(title='用户群体')
plt.show()

# 分析各群体特征
cluster_analysis = user_features.groupby('cluster').agg({
    'total_purchases': 'mean',
    'total_amount': 'mean',
    'avg_order_value': 'mean',
    'repurchase_label': 'mean',
    'user_id': 'count'
}).rename(columns={'user_id': '用户数'})

print('各用户群体特征分析:')
print(cluster_analysis.round(2))

In [ ]:
# ========================================
# 第十部分：REST API接口示例
# ========================================

# 创建预测函数
def predict_repurchase(user_data):
    """
    预测用户复购概率
    
    参数:
        user_data: 用户特征字典
    
    返回:
        复购概率
    """
    # 创建特征数组
    features = np.array([[
        user_data.get('total_clicks', 0),
        user_data.get('total_browses', 0),
        user_data.get('total_purchases', 0),
        user_data.get('total_amount', 0),
        user_data.get('avg_price', 0),
        user_data.get('purchase_days', 0),
        user_data.get('unique_categories', 0),
        user_data.get('unique_products', 0),
        user_data.get('avg_order_value', 0),
        user_data.get('click_to_purchase_ratio', 0),
        user_data.get('browse_to_purchase_ratio', 0),
        user_data.get('days_since_last_action', 30),
        user_data.get('active_days', 0),
        user_data.get('age', 30)
    ]])
    
    # 标准化
    features_scaled = scaler.transform(features)
    
    # 预测
    probability = model.predict(features_scaled)[0][0]
    
    return float(probability)

# 测试示例
test_user = {
    'total_clicks': 50,
    'total_browses': 200,
    'total_purchases': 10,
    'total_amount': 2500,
    'avg_price': 250,
    'purchase_days': 5,
    'unique_categories': 5,
    'unique_products': 15,
    'avg_order_value': 250,
    'click_to_purchase_ratio': 5,
    'browse_to_purchase_ratio': 20,
    'days_since_last_action': 5,
    'active_days': 30,
    'age': 28
}

prob = predict_repurchase(test_user)
print(f'测试用户复购概率: {prob:.2%}')

---

## 项目总结

本项目完成了以下内容：

1. **数据模拟生成** - 生成了5000用户、50000条行为记录
2. **特征工程** - 构建了14个用户行为特征
3. **标签构造** - 定义了30天复购标签
4. **传统机器学习** - 逻辑回归和随机森林模型
5. **深度学习** - DNN神经网络模型
6. **模型对比** - 准确率、AUC、召回率评估
7. **用户分群** - K-Means聚类分析
8. **API接口** - 实时复购概率预测函数

---

**课程设计完成!**